# AgentCore Observability Lab
## Mastering Amazon Bedrock AgentCore | Pumping Code

---

## What You'll Build

In this lab you will deploy a small **runtime-hosted Strands agent** and use **Amazon Bedrock AgentCore Observability** to inspect how requests flow through sessions, traces, spans, and CloudWatch logs.

By the end of this lab, you will have:
- Deployed a Strands agent to AgentCore Runtime with automatic observability enabled
- Invoked the same runtime session multiple times to build a session timeline
- Sent a request with explicit distributed tracing context (`traceParent`, `traceState`, `baggage`)
- Queried observability data programmatically
- Located the same runtime in CloudWatch GenAI Observability, Transaction Search, and CloudWatch Logs
- Cleaned up the runtime and local lab files

---

## Architecture

```
┌──────────────────────────────────────────────┐
│          Strands Runtime-Hosted Agent        │
│ Claude Haiku 4.5 + Calculator + Weather Tool │
└──────────────┬───────────────────────────────┘
               │  invoke_agent_runtime
               ▼
┌──────────────────────────────────────────────┐
│          AgentCore Runtime Session           │
│  repeated calls share runtimeSessionId       │
└──────────────┬───────────────────────────────┘
               │  automatic OTEL traces
               ▼
┌──────────────────────────────────────────────┐
│      CloudWatch GenAI Observability          │
│  Agents View → Sessions View → Trace View    │
└──────────────┬───────────────────────────────┘
               │
               ├── Transaction Search: /aws/spans/default
               └── Runtime Log Groups
```


## Prerequisites
- AWS CLI configured with appropriate credentials
- Python 3.10+
- Access to Amazon Bedrock, ECR, and AgentCore Runtime
- Bedrock access enabled for: `us.anthropic.claude-sonnet-4-6`
- CloudWatch access
- CloudWatch **Transaction Search** enabled in this account and region

This lab runs TypeScript on the Deno kernel. Pick the **Deno** kernel in the top right.


---
# Part 1: Environment Setup

AgentCore Runtime-hosted agents are instrumented automatically. The remaining setup work in this notebook is:

1. Install the runtime and observability toolkit packages
2. Deploy a small Strands agent
3. Generate session and trace data through controlled invocations
4. Verify observability data programmatically

> **Important**: If Transaction Search is not enabled yet, the runtime still works, but traces and spans might not show up in CloudWatch for several minutes.


In [ ]:
// Dependencies are pinned in the project's deno.json and cached by ./setup.sh; there is no install
// step. (The Python lab installed bedrock-agentcore, the starter toolkit, boto3 and strands here.)
import { sh } from "../shared/notebook.ts";
await sh("deno", ["--version"]);

In [ ]:
Deno.env.set("AWS_REGION", "us-east-1");
const region = Deno.env.get("AWS_REGION")!;

// APPROACH A: Use credentials
// Deno.env.set("AWS_ACCESS_KEY_ID", "your_access_key");
// APPROACH B: Use AWS SSO profile
// Deno.env.set("AWS_PROFILE", "your_profile");

console.log(`\u2705 Region: ${region}`);

In [ ]:
import { GetCallerIdentityCommand, STSClient } from "@aws-sdk/client-sts";
import { loadEnv, state, writeFile } from "../shared/notebook.ts";

await loadEnv();

const identity = await new STSClient({ region }).send(new GetCallerIdentityCommand({}));
console.log(`\u2705 AWS account: ${identity.Account}`);

---
# Part 2: Define the Runtime-Hosted Agent

This lab uses a deliberately small agent:
- `calculator` for deterministic math
- `get_weather` as a simple custom tool
- Claude Haiku 4.5 as the runtime model

The point of this notebook is not complex agent behavior. The point is to generate a runtime that is easy to invoke, easy to reason about, and easy to inspect in observability tooling.


In [ ]:
// Write the lab agent and its dependency file, replacing the Python lab's file-writing cell.
import { dirname, fromFileUrl, join } from "@std/path";

const PROJECT_DIR = join(dirname(fromFileUrl(import.meta.url ?? "file://" + Deno.cwd() + "/x")), "..");
const OBSERVABILITY_DIR = "../backend/observability";

const AGENT_CODE = `import { Agent, BedrockModel, tool } from "@strands-agents/sdk";
import { BedrockAgentCoreApp } from "bedrock-agentcore/runtime";
import { z } from "zod";

const MODEL_ID = "us.anthropic.claude-sonnet-4-6";

// Strands TypeScript ships no \`calculator\`, so the lab brings its own.
const calculator = tool({
  name: "calculator",
  description: "Evaluate an arithmetic expression, e.g. '18 * 7'.",
  inputSchema: z.object({ expression: z.string() }),
  callback: ({ expression }) => {
    if (!/^[\\d\\s+\\-*/().]+$/.test(expression)) return "invalid expression";
    return String(Function(\`"use strict"; return (\${expression});\`)());
  },
});

const getWeather = tool({
  name: "get_weather",
  description: "Return a mock weather payload for observability demonstrations.",
  inputSchema: z.object({ location: z.string() }),
  callback: ({ location }) => ({
    location,
    temperature_celsius: 22,
    condition: "Partly cloudy",
    humidity_percent: 65,
  }),
});

const model = new BedrockModel({ modelId: MODEL_ID });

const agent = new Agent({
  model,
  tools: [calculator, getWeather],
  systemPrompt: "You are a compact observability demo agent. " +
    "Use calculator for arithmetic and get_weather for weather lookups. " +
    "Be concise and explicit about when a tool was used.",
});

const app = new BedrockAgentCoreApp({
  invocationHandler: {
    requestSchema: z.object({ prompt: z.string().default("") }),
    process: async ({ prompt }) => (await agent.invoke(prompt)).toString(),
  },
});

app.run();
`;

// The agent's own deno.json plays the role of the Python lab's requirements file.
const rootImports = JSON.parse(await Deno.readTextFile("../../deno.json")).imports;
const AGENT_CONFIG = JSON.stringify(
  {
    nodeModulesDir: "auto",
    compilerOptions: { strict: true },
    imports: Object.fromEntries(
      [
        "@strands-agents/sdk",
        "bedrock-agentcore/",
        "zod",
        "@opentelemetry/api",
        "@opentelemetry/api-logs",
        "@opentelemetry/context-async-hooks",
        "@opentelemetry/core",
        "@opentelemetry/otlp-transformer",
        "@opentelemetry/resources",
        "@opentelemetry/sdk-logs",
        "@opentelemetry/sdk-trace-base",
        "@smithy/protocol-http",
        "@smithy/signature-v4",
        "@aws-crypto/sha256-js",
        "@aws-sdk/credential-provider-node",
      ].map((k) => [k, rootImports[k]]),
    ),
  },
  null,
  2,
) + "\n";

await writeFile(`${OBSERVABILITY_DIR}/observability_lab_agent.ts`, AGENT_CODE);
await writeFile(`${OBSERVABILITY_DIR}/deno.json`, AGENT_CONFIG);

---
# Part 3: Deploy to AgentCore Runtime

AgentCore Runtime-hosted agents are automatically instrumented for observability. Once the runtime reaches `READY`, every invocation in this notebook will emit telemetry that can be inspected in:

- CloudWatch GenAI Observability
- Transaction Search at `/aws/spans/default`
- Runtime log groups


In [ ]:
import { Runtime } from "../toolkit/mod.ts";

const AGENT_NAME = "observability_lab_agent";

const agentcoreRuntime = new Runtime();
await agentcoreRuntime.configure({
  entrypoint: "observability_lab_agent.ts",
  autoCreateExecutionRole: true,
  autoCreateEcr: true,
  requirementsFile: "deno.json",
  region,
  agentName: AGENT_NAME,
  sourceDir: OBSERVABILITY_DIR,
});

console.log("\ud83d\ude80 Launching the observability lab agent...");
const launchResult = await agentcoreRuntime.launch();
console.log(`\u2705 Deployed: ${launchResult.agentId}`);
console.log(`   ARN: ${launchResult.agentArn}`);

In [ ]:
console.log("\u23f3 Waiting for runtime to reach READY...");
const endStatuses = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"];
let status = "";
while (!endStatuses.includes(status)) {
  const statusResponse = await agentcoreRuntime.status();
  status = (statusResponse.endpoint as { status?: string })?.status ?? "UNKNOWN";
  if (!endStatuses.includes(status)) {
    console.log(`   status: ${status}`);
    await new Promise((r) => setTimeout(r, 15_000));
  }
}
console.log(`\u2705 Runtime status: ${status}`);

In [ ]:
const primarySessionId = `obs-lab-${crypto.randomUUID()}-${crypto.randomUUID().slice(0, 8)}`;

const labConfig: Record<string, unknown> = {
  runtime: {
    agent_name: AGENT_NAME,
    agent_id: launchResult.agentId,
    agent_arn: launchResult.agentArn,
    region,
  },
  primary_session_id: primarySessionId,
};

console.log(`\ud83e\uddea Session id for this lab: ${primarySessionId}`);

---
# Part 4: Generate Session and Trace Data

All three invocations reuse the same `runtimeSessionId`. This is the key pattern for session-aware observability:

- one session
- multiple related requests
- multiple traces and spans tied back to the same session

The third invocation keeps the same session and adds distributed tracing context. In practice, the most reliable way to inspect traces in this lab is to derive them from the queried spans in CloudWatch.


In [ ]:
import { BedrockAgentCoreClient, InvokeAgentRuntimeCommand } from "@aws-sdk/client-bedrock-agentcore";

const agentcoreClient = new BedrockAgentCoreClient({ region });

interface InvocationRecord {
  prompt: string;
  runtimeSessionId?: string;
  traceId?: string;
  traceParent?: string;
  statusCode?: number;
  body_preview: string;
}

async function invokeRuntime(
  prompt: string,
  sessionId: string,
  extra: { traceParent?: string; traceState?: string; baggage?: string } = {},
): Promise<InvocationRecord> {
  const response = await agentcoreClient.send(
    new InvokeAgentRuntimeCommand({
      agentRuntimeArn: launchResult.agentArn,
      runtimeSessionId: sessionId,
      payload: new TextEncoder().encode(JSON.stringify({ prompt })),
      contentType: "application/json",
      accept: "application/json",
      qualifier: "DEFAULT",
      ...extra,
    }),
  );
  const body = await response.response!.transformToString();
  const record: InvocationRecord = {
    prompt,
    runtimeSessionId: response.runtimeSessionId,
    traceId: response.traceId,
    traceParent: response.traceParent,
    statusCode: response.statusCode,
    body_preview: body.slice(0, 500),
  };
  console.log(`Prompt: ${prompt}`);
  console.log(`  session:   ${record.runtimeSessionId}`);
  console.log(`  traceId:   ${record.traceId}`);
  console.log(`  status:    ${record.statusCode}`);
  console.log(`  response:  ${record.body_preview.slice(0, 160)}`);
  return record;
}

In [ ]:
const firstInvocation = await invokeRuntime("What is 18 * 7? Explain briefly.", primarySessionId);

const secondInvocation = await invokeRuntime(
  "What is the weather in Berlin right now?",
  primarySessionId,
);

const customTraceParent = `00-${crypto.randomUUID().replaceAll("-", "")}-${
  crypto.randomUUID().replaceAll("-", "").slice(0, 16)
}-01`;
const thirdInvocation = await invokeRuntime(
  "Add 144 and 256, then tell me whether you used a tool.",
  primarySessionId,
  {
    traceParent: customTraceParent,
    traceState: "vendor=pumping-code-lab",
    baggage: "course=mastering-agentcore,chapter=09",
  },
);

labConfig.invocations = [firstInvocation, secondInvocation, thirdInvocation];

console.log("\n\u2705 Runtime invocations complete");

---
# Part 5: Query Observability Data Programmatically

This notebook uses the toolkit's `ObservabilityClient` path to verify that spans are visible for the generated runtime session.

This section performs three checks:

1. Wait for spans to become visible for the shared session
2. Summarize the traces and spans recorded under that session
3. Resolve the runtime log groups in CloudWatch Logs


In [ ]:
import { ObservabilityClient } from "../toolkit/mod.ts";

async function waitForSessionSpans(
  agentId: string,
  sessionId: string,
  attempts = 12,
  delaySeconds = 20,
  lookbackDays = 1,
) {
  const obsClient = new ObservabilityClient({ region });
  for (let attempt = 1; attempt <= attempts; attempt++) {
    const endTime = Date.now();
    const startTime = endTime - lookbackDays * 24 * 60 * 60 * 1000;
    const spans = await obsClient.querySpansBySession({
      sessionId,
      startTimeMs: startTime,
      endTimeMs: endTime,
      agentId,
    });
    if (spans.length > 0) {
      console.log(`\u2705 Found ${spans.length} spans for session ${sessionId}`);
      return spans;
    }
    if (attempt === attempts) {
      throw new Error(
        `No spans found for session ${sessionId} after ${attempts} checks. ` +
          "Check whether CloudWatch Transaction Search is enabled and whether spans have finished ingesting.",
      );
    }
    console.log(`   Spans not visible yet (attempt ${attempt}/${attempts}). Waiting ${delaySeconds}s...`);
    await new Promise((r) => setTimeout(r, delaySeconds * 1000));
  }
  return [];
}

const primarySpans = await waitForSessionSpans(launchResult.agentId, primarySessionId);

function summarizeSpans(label: string, spans: { traceId?: string; spanName?: string }[]) {
  const traceIds = [...new Set(spans.map((s) => s.traceId).filter(Boolean))].sort();
  const spanNames = [...new Set(spans.map((s) => s.spanName).filter(Boolean))].sort();
  console.log(`\n${label}`);
  console.log(`  span count:  ${spans.length}`);
  console.log(`  trace count: ${traceIds.length}`);
  console.log(`  sample traces: ${JSON.stringify(traceIds.slice(0, 5))}`);
  console.log(`  sample spans:  ${JSON.stringify(spanNames.slice(0, 8))}`);
}

summarizeSpans("Shared session summary", primarySpans);

In [ ]:
import { CloudWatchLogsClient, DescribeLogGroupsCommand } from "@aws-sdk/client-cloudwatch-logs";

const logsClient = new CloudWatchLogsClient({ region });
const logGroups = [];
let nextToken: string | undefined;
do {
  const page = await logsClient.send(
    new DescribeLogGroupsCommand({
      logGroupNamePrefix: `/aws/bedrock-agentcore/runtimes/${launchResult.agentId}`,
      nextToken,
    }),
  );
  logGroups.push(...(page.logGroups ?? []));
  nextToken = page.nextToken;
} while (nextToken);

if (logGroups.length === 0) {
  throw new Error("No CloudWatch log groups found for the deployed runtime.");
}

console.log("\u2705 CloudWatch log groups discovered for this runtime:");
for (const group of logGroups) {
  console.log(`   - ${group.logGroupName}`);
}

---
# Part 6: Where to Inspect the Same Data in CloudWatch

Use the identifiers printed in this notebook to inspect the runtime manually in CloudWatch:

- **GenAI Observability → Agents View**
  - Filter by the runtime name or `agent_id`
  - Use this to see the top-level runtime and aggregate health
- **Sessions View**
  - Open the session that matches `primary_session_id`
  - You should see multiple invocations tied to the same session
- **Trace View**
  - Open any trace recorded under that session
  - Inspect model calls, tool calls, and timing
- **CloudWatch Logs**
  - Standard logs: `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint>/[runtime-logs]...`
  - OTEL logs: `/aws/bedrock-agentcore/runtimes/<agent_id>-<endpoint>/otel-rt-logs`
- **Transaction Search**
  - Search path: `/aws/spans/default`
  - Filter by trace ID, service name, or session-related identifiers

The invocation API may also return tracing fields:
- `traceId`
- `traceParent`
- `traceState`
- `baggage`

In this lab, the span query is the more reliable source for trace IDs when correlating notebook output with CloudWatch Transaction Search.


In [ ]:
console.log("CloudWatch lookup values");
console.log("------------------------");
console.log(`Agent name:              ${AGENT_NAME}`);
console.log(`Agent ID:                ${launchResult.agentId}`);
console.log(`Agent ARN:               ${launchResult.agentArn}`);
const traceIds = [...new Set(primarySpans.map((s) => s.traceId).filter(Boolean))].sort();
console.log(`Session ID:              ${primarySessionId}`);
console.log(`Trace IDs from spans:    ${JSON.stringify(traceIds)}`);
console.log();
console.log("Console paths");
console.log("-------------");
console.log("CloudWatch \u2192 GenAI Observability \u2192 Bedrock AgentCore \u2192 Agents");
console.log("CloudWatch \u2192 Transaction Search \u2192 /aws/spans/default");
console.log("CloudWatch \u2192 Logs \u2192 Log groups");

await writeFile("environments/observability_lab_config.json", `${JSON.stringify(labConfig, null, 2)}\n`);
await state.set("observability_lab", labConfig as never);

---
# Part 7: Cleanup

This cleanup removes:
- the runtime deployment
- the generated local config file
- the local runtime metadata file

CloudWatch traces and logs remain in your AWS account for later inspection according to your retention settings.


In [ ]:
import { BedrockAgentCoreControlClient, DeleteAgentRuntimeCommand, GetAgentRuntimeCommand } from "@aws-sdk/client-bedrock-agentcore-control";

const controlClient = new BedrockAgentCoreControlClient({ region });

async function deleteRuntimeAndWait(agentId: string, attempts = 40, delaySeconds = 15) {
  try {
    await controlClient.send(new DeleteAgentRuntimeCommand({ agentRuntimeId: agentId }));
    console.log(`\ud83d\uddd1\ufe0f  Delete requested for runtime ${agentId}`);
  } catch (error) {
    if ((error as Error).name === "ResourceNotFoundException") {
      console.log(`\u2139\ufe0f  Runtime ${agentId} already deleted`);
      return;
    }
    throw error;
  }

  for (let attempt = 1; attempt <= attempts; attempt++) {
    try {
      const response = await controlClient.send(new GetAgentRuntimeCommand({ agentRuntimeId: agentId }));
      console.log(`   Runtime status: ${response.status}`);
    } catch (error) {
      if ((error as Error).name === "ResourceNotFoundException") {
        console.log("\u2705 Runtime deleted");
        return;
      }
      throw error;
    }
    if (attempt === attempts) throw new Error(`Runtime ${agentId} still exists after waiting for deletion.`);
    await new Promise((r) => setTimeout(r, delaySeconds * 1000));
  }
}

await deleteRuntimeAndWait(launchResult.agentId);

for (const path of [
  `${OBSERVABILITY_DIR}/observability_lab_agent.ts`,
  `${OBSERVABILITY_DIR}/deno.json`,
  `${OBSERVABILITY_DIR}/Dockerfile`,
]) {
  try {
    await Deno.remove(path);
    console.log(`\ud83e\uddf9 Removed local file: ${path}`);
  } catch {
    // already gone
  }
}